# Notebook 1 - Exploración del Low Carbon London Dataset

Objetivo: entender el comportamiento del consumo eléctrico y justificar un problema de forecasting.

## 1. Librerías y configuración

In [ ]:
import zipfile
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10,5)

## 2. Carga de una muestra del dataset

In [ ]:
ZIP_PATH = '../data/LCL_Data.zip'

with zipfile.ZipFile(ZIP_PATH) as z:
    files = z.namelist()

print('Archivos encontrados:', len(files))
print(files[:5])

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(files[0]) as f:
        df = pd.read_csv(f, nrows=1_000_000)

df.head()

## 3. Limpieza y conversión de tipos

In [ ]:
df.columns = df.columns.str.strip()
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['KWH/hh (per half hour)'] = pd.to_numeric(df['KWH/hh (per half hour)'], errors='coerce')

df.info()

## 4. Información general

In [ ]:
print('Hogares únicos:', df['LCLid'].nunique())
print('\nTipos de tarifa:')
print(df['stdorToU'].value_counts())
print('\nRango temporal:')
print(df['DateTime'].min(), '->', df['DateTime'].max())
print('\nConsumo:')
print(df['KWH/hh (per half hour)'].describe())

## 5. Distribución del consumo

In [ ]:
df['KWH/hh (per half hour)'].hist(bins=100)
plt.title('Distribución del consumo')
plt.xlabel('kWh')
plt.ylabel('Frecuencia')
plt.show()

display(df['KWH/hh (per half hour)'].quantile([0.5,0.9,0.95,0.99]))

## 6. Patrón horario

In [ ]:
df['hour'] = df['DateTime'].dt.hour

df.groupby('hour')['KWH/hh (per half hour)'].mean().plot()
plt.title('Consumo promedio por hora')
plt.show()

## 7. Patrón semanal

In [ ]:
df['weekday'] = df['DateTime'].dt.dayofweek

df.groupby('weekday')['KWH/hh (per half hour)'].mean().plot(kind='bar')
plt.title('Consumo promedio por día de la semana')
plt.show()

## 8. Serie temporal de un hogar

In [ ]:
hogar = df[df['LCLid'] == df['LCLid'].iloc[0]].sort_values('DateTime')

hogar.iloc[:1000].plot(
    x='DateTime',
    y='KWH/hh (per half hour)',
    figsize=(15,5)
)
plt.title('20 días aproximados de un hogar')
plt.show()

## 9. Descubrimiento principal

Hallazgos:

- Existe un patrón horario fuerte.
- Existe dependencia temporal entre observaciones consecutivas.
- El consumo presenta periodicidad diaria.
- La distribución es asimétrica con pocos picos altos.
- El problema de forecasting está justificado.

Problema elegido:

**Predecir el consumo de la siguiente media hora usando información histórica.**